# 13 — Animated and Storytelling Visuals

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook creates animated and presentation-style visuals. It uses Plotly animation for data-driven movement and an Anime.js-powered HTML storytelling page for presentation use.


## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("13_animated_storytelling_visuals", total_steps=5, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

Animations are used only where movement adds meaning: candidate attention over time, hashtag growth over time, and a final story page explaining how the project connects online networks to actual election outcomes.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]
print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
# Load processed entity data. Run notebooks 01–05 before this notebook.
tweets_path = PROCESSED / "tweets_with_entities.csv"
if not tweets_path.exists():
    raise FileNotFoundError("Run 04_entity_extraction_candidate_dictionary.ipynb first to create tweets_with_entities.csv")

tweets = pd.read_csv(tweets_path, parse_dates=["createdAt"])
tweets = parse_list_columns(tweets, ["hashtags", "candidates_mentioned"])
tweets["date"] = pd.to_datetime(tweets["date"], errors="coerce")
print(tweets.shape)


In [ ]:
progress.step("Purpose")
# Animated candidate attention race: cumulative mentions over time
rows = []
for _, row in tweets.iterrows():
    for c in row["candidates_mentioned"]:
        rows.append({"date": row["date"], "candidate": c, "count": 1})

candidate_daily = pd.DataFrame(rows)
if candidate_daily.empty:
    print("No candidate mentions available for animation.")
else:
    candidate_daily = candidate_daily.groupby(["date", "candidate"], as_index=False)["count"].sum()
    top_candidates = candidate_daily.groupby("candidate")["count"].sum().sort_values(ascending=False).head(15).index.tolist()
    anim_df = candidate_daily[candidate_daily["candidate"].isin(top_candidates)].copy()
    # Build complete date-candidate grid for smooth cumulative animation
    dates = pd.date_range(anim_df["date"].min(), anim_df["date"].max(), freq="D")
    grid = pd.MultiIndex.from_product([dates, top_candidates], names=["date", "candidate"]).to_frame(index=False)
    anim_df = grid.merge(anim_df, on=["date", "candidate"], how="left").fillna({"count": 0})
    anim_df["cumulative_mentions"] = anim_df.groupby("candidate")["count"].cumsum()
    anim_df["date_label"] = anim_df["date"].dt.strftime("%Y-%m-%d")
    anim_df["rank"] = anim_df.groupby("date")["cumulative_mentions"].rank(method="first", ascending=False)
    anim_df = anim_df[anim_df["rank"] <= 12]

    fig = px.bar(
        anim_df.sort_values(["date", "cumulative_mentions"]),
        x="cumulative_mentions",
        y="candidate",
        orientation="h",
        animation_frame="date_label",
        animation_group="candidate",
        range_x=[0, max(1, anim_df["cumulative_mentions"].max() * 1.08)],
        title="Animated candidate attention race: cumulative Twitter/X mentions",
        labels={"cumulative_mentions": "Cumulative mentions", "candidate": "Candidate"},
    )
    fig.update_layout(height=780, yaxis={"categoryorder": "total ascending"})
    save_plotly(fig, INTERACTIVE / "13_animated_candidate_attention_race.html")
    fig.show()


In [ ]:
progress.step("Purpose")
# Animated hashtag growth: top hashtags over time
rows = []
for _, row in tweets.iterrows():
    for h in row["hashtags"]:
        rows.append({"date": row["date"], "hashtag": h, "count": 1})

tag_daily = pd.DataFrame(rows)
if tag_daily.empty:
    print("No hashtags available for animation.")
else:
    tag_daily = tag_daily.groupby(["date", "hashtag"], as_index=False)["count"].sum()
    top_hashtags = tag_daily.groupby("hashtag")["count"].sum().sort_values(ascending=False).head(15).index.tolist()
    anim_tags = tag_daily[tag_daily["hashtag"].isin(top_hashtags)].copy()
    dates = pd.date_range(anim_tags["date"].min(), anim_tags["date"].max(), freq="D")
    grid = pd.MultiIndex.from_product([dates, top_hashtags], names=["date", "hashtag"]).to_frame(index=False)
    anim_tags = grid.merge(anim_tags, on=["date", "hashtag"], how="left").fillna({"count": 0})
    anim_tags["cumulative_uses"] = anim_tags.groupby("hashtag")["count"].cumsum()
    anim_tags["date_label"] = anim_tags["date"].dt.strftime("%Y-%m-%d")
    anim_tags["rank"] = anim_tags.groupby("date")["cumulative_uses"].rank(method="first", ascending=False)
    anim_tags = anim_tags[anim_tags["rank"] <= 12]

    fig = px.bar(
        anim_tags.sort_values(["date", "cumulative_uses"]),
        x="cumulative_uses",
        y="hashtag",
        orientation="h",
        animation_frame="date_label",
        animation_group="hashtag",
        range_x=[0, max(1, anim_tags["cumulative_uses"].max() * 1.08)],
        title="Animated hashtag growth: cumulative uses over time",
        labels={"cumulative_uses": "Cumulative uses", "hashtag": "Hashtag"},
    )
    fig.update_layout(height=780, yaxis={"categoryorder": "total ascending"})
    save_plotly(fig, INTERACTIVE / "13_animated_hashtag_growth.html")
    fig.show()


In [ ]:
progress.step("Purpose")
# Anime.js storytelling page.
# This file becomes a lightweight presentation page that links to the generated outputs.
# It uses a CDN for anime.js, so it animates when opened with internet access.

story_html = f"""
<!doctype html>
<html lang='en'>
<head>
<meta charset='utf-8'>
<meta name='viewport' content='width=device-width, initial-scale=1'>
<title>Philippine Election 2025 Network Observatory</title>
<script src='https://cdnjs.cloudflare.com/ajax/libs/animejs/3.2.2/anime.min.js'></script>
<style>
  body {{ margin:0; font-family: Inter, Segoe UI, Arial, sans-serif; background:#0f172a; color:#e5e7eb; }}
  .hero {{ min-height:90vh; display:flex; align-items:center; justify-content:center; padding:48px; background: radial-gradient(circle at top left, #1d4ed8, transparent 35%), radial-gradient(circle at bottom right, #facc15, transparent 25%), #0f172a; }}
  .panel {{ max-width:1100px; background:rgba(15,23,42,.72); border:1px solid rgba(255,255,255,.14); box-shadow:0 30px 90px rgba(0,0,0,.35); border-radius:28px; padding:46px; backdrop-filter: blur(16px); }}
  h1 {{ font-size: clamp(42px, 7vw, 86px); line-height:.95; margin:0 0 24px; letter-spacing:-.05em; }}
  .subtitle {{ font-size: clamp(18px, 2vw, 25px); color:#cbd5e1; max-width:900px; }}
  .grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(240px,1fr)); gap:18px; margin-top:34px; }}
  .card {{ padding:24px; border-radius:22px; background:rgba(255,255,255,.08); border:1px solid rgba(255,255,255,.12); transform:translateY(20px); opacity:0; }}
  .card b {{ color:#facc15; }}
  section {{ padding:70px 48px; max-width:1180px; margin:auto; }}
  a {{ color:#93c5fd; text-decoration:none; }}
  .asset {{ background:#111827; border:1px solid rgba(255,255,255,.12); border-radius:18px; padding:20px; margin:12px 0; }}
</style>
</head>
<body>
  <div class='hero'>
    <div class='panel'>
      <h1>Philippine Election 2025<br>Network Observatory</h1>
      <p class='subtitle'>A network science study of Twitter/X election discourse and actual Senate election outcomes: hashtags, candidates, communities, centrality, and online-outcome alignment.</p>
      <div class='grid'>
        <div class='card'><b>Communication analytics</b><br>Tweet volume, engagement, language, verified activity, and candidate visibility.</div>
        <div class='card'><b>Network science</b><br>Hashtag, candidate co-mention, candidate-hashtag, mention, and reply networks.</div>
        <div class='card'><b>Election outcome layer</b><br>Actual Senate vote rank, vote share, regional strength, and winner labels.</div>
        <div class='card'><b>Alignment test</b><br>Correlation, top-12 overlap, rank difference, and online overperformance.</div>
      </div>
    </div>
  </div>
  <section>
    <h2>Generated interactive assets</h2>
    <div class='asset'><a href='13_animated_candidate_attention_race.html'>Animated candidate attention race</a></div>
    <div class='asset'><a href='13_animated_hashtag_growth.html'>Animated hashtag growth</a></div>
    <div class='asset'><a href='06_hashtag_network_interactive.html'>Interactive hashtag co-occurrence network</a></div>
    <div class='asset'><a href='07_candidate_comention_network_interactive.html'>Interactive candidate co-mention network</a></div>
    <div class='asset'><a href='08_candidate_hashtag_bipartite_interactive.html'>Interactive candidate-hashtag bipartite network</a></div>
    <div class='asset'><a href='09_user_mention_network_interactive.html'>Interactive user mention network</a></div>
  </section>
<script>
anime({{ targets: '.panel', translateY: [30,0], opacity: [0,1], duration: 1200, easing: 'easeOutExpo' }});
anime({{ targets: '.card', translateY: [35,0], opacity: [0,1], delay: anime.stagger(180, {{start: 500}}), duration: 900, easing: 'easeOutExpo' }});
</script>
</body>
</html>
"""

out = INTERACTIVE / "13_network_observatory_story_page.html"
out.write_text(story_html, encoding="utf-8")
print("Saved Anime.js storytelling page:", out)


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
